# 01 — Feature engineering on C-MAPSS

We start from the raw sensor table that notebook 00 explored and produce a **gold** feature table ready for tabular models (notebook 02). The pipeline:

1. Drop constant / non-informative sensors.
2. Per-sensor rolling stats (mean / std / min / max / slope) over windows {5, 15, 30}.
3. FFT band-energy features (low / mid / high) over a 30-cycle window.
4. CUSUM change-point features.
5. Persist the result as Parquet at `data/processed/gold_FD001.parquet` for downstream notebooks.

All of this lives in `turboguard.features.pipeline` so the same pipeline can be re-run on any of the four sub-datasets.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from turboguard.data.cmapss import load_cmapss
from turboguard.features.pipeline import (
    FeatureConfig,
    build_features,
    find_constant_sensors,
    save_gold,
)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
GOLD_DIR = ROOT / 'data' / 'processed'

## 1. Load FD001 and add per-row RUL

In [ ]:
fd001 = load_cmapss('FD001')
df = fd001.with_rul()
print(f'FD001 train rows: {len(df):,} | engines: {df.unit_id.nunique()}')
df.head()

## 2. Identify constant sensors

These contribute nothing and inflate the feature space. The pipeline drops them automatically — this is a sanity-check view of which ones.

In [ ]:
sensor_cols = [c for c in df.columns if c.startswith('sensor_')]
constant = find_constant_sensors(df, sensor_cols)
kept_initial = [c for c in sensor_cols if c not in constant]
print(f'Constant sensors dropped: {constant}')
print(f'Kept           : {len(kept_initial)} sensors')

## 3. Run the full feature pipeline

This computes ~170 features per row. On FD001 (~20k rows) it takes a couple of minutes — the FFT and change-point loops are pure Python for clarity. If we ever need it faster, both can be Numba-compiled trivially.

In [ ]:
config = FeatureConfig(
    rolling_windows=(5, 15, 30),
    fft_window=30,
    cusum_window=15,
    cusum_threshold=2.5,
)

t0 = time.time()
gold, kept_sensors = build_features(df, config=config)
elapsed = time.time() - t0
print(f'Built {gold.shape[1]} features over {len(gold):,} rows in {elapsed:.1f}s')
print(f'Surviving sensors: {len(kept_sensors)}')

## 4. Quick sanity: does the FFT high/low ratio rise as engines approach failure?

Pick the most variable sensor and plot the high-to-low band ratio against RUL. We expect the ratio to grow as RUL drops.

In [ ]:
# Pick the sensor with the strongest absolute correlation between mean_30 and RUL.
candidates = [(c, gold[f'{c}_mean_30'].corr(gold['RUL'])) for c in kept_sensors]
candidates.sort(key=lambda kv: abs(kv[1]), reverse=True)
top_sensor, corr = candidates[0]
print(f'Most degrading sensor: {top_sensor}  (corr(mean_30, RUL) = {corr:+.3f})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for uid in [1, 50, 100]:
    sub = gold[gold.unit_id == uid].sort_values('cycle')
    axes[0].plot(sub.cycle, sub[f'{top_sensor}_mean_30'], alpha=0.7, label=f'unit {uid}')
    axes[1].plot(sub.cycle, sub[f'{top_sensor}_fft_hi_lo_30'], alpha=0.7, label=f'unit {uid}')
axes[0].set_title(f'{top_sensor}_mean_30 vs cycle')
axes[1].set_title(f'{top_sensor}_fft_hi_lo_30 vs cycle (degradation proxy)')
for ax in axes:
    ax.set_xlabel('cycle'); ax.legend()
fig.tight_layout()

## 5. Top-15 features by absolute correlation with RUL

A correlation pass is not a substitute for a real model, but it tells us whether the feature engineering is producing signal.

In [ ]:
feature_cols = [c for c in gold.columns if c not in {'unit_id', 'cycle', 'RUL'} and gold[c].dtype.kind in 'fi']
corrs = gold[feature_cols].corrwith(gold['RUL']).sort_values(key=lambda s: s.abs(), ascending=False)
corrs.head(15).to_frame('corr_with_RUL')

## 6. Persist gold features for downstream notebooks

In [ ]:
out_path = save_gold(gold, GOLD_DIR / 'gold_FD001.parquet')
print(f'Saved {len(gold):,} rows × {gold.shape[1]} cols to {out_path}')

---

## Next: `02_baselines_xgb_lgbm.ipynb`

We'll load `gold_FD001.parquet`, train XGBoost and LightGBM RUL regressors with engine-stratified validation, and log every run to MLflow with model signatures.